In [1]:
chr(0)

'\x00'

In [5]:
print(chr(0))

 


In [7]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
 return "".join([bytes([b]).decode("utf-8") for b in bytestring])
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [8]:
class Demo:
    value = 10

    @classmethod
    def add_cls(cls, x):
        return cls.value + x        # ✅ 可以访问 cls.value

    @staticmethod
    def add_static(x):
        return x + 5                # ❌ 不能访问 Demo.value

In [10]:
print(Demo.add_cls(5))

15


In [36]:
def sub_func():
    return [1, 2, 3]  # 普通返回

def main_gen():
    yield sub_func()  # ✅ 可以！因为 list 是可迭代的

for i in main_gen():
    print(i)

[1, 2, 3]


In [1]:
from torch import nn
import torch
class Demo(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.linear=nn.Linear(d_in,d_out)
    def forward(self,in_features):
        return self.linear(in_features)

num=torch.randn(5,10)
A=Demo(10,5)
B=A(num)
print(type(B))

<class 'torch.Tensor'>


In [43]:
import torch
A=torch.range(1,27)
A=A.unsqueeze(0)
A=A.view(3,3,3)
B=torch.tensor([2]*3)
print(A)
A=A*B
print(A)

tensor([[[ 1.,  2.,  3.],
         [ 4.,  5.,  6.],
         [ 7.,  8.,  9.]],

        [[10., 11., 12.],
         [13., 14., 15.],
         [16., 17., 18.]],

        [[19., 20., 21.],
         [22., 23., 24.],
         [25., 26., 27.]]])
tensor([[[ 2.,  4.,  6.],
         [ 8., 10., 12.],
         [14., 16., 18.]],

        [[20., 22., 24.],
         [26., 28., 30.],
         [32., 34., 36.]],

        [[38., 40., 42.],
         [44., 46., 48.],
         [50., 52., 54.]]])


C:\Users\谌伦杰\AppData\Local\Temp\ipykernel_70412\2561022888.py:2: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  A=torch.range(1,27)


In [102]:
import numpy
A=torch.randint(0,10,(5,))
print(A)
B=torch.randint(0,10,(5,))
print(B)
C=torch.stack((A,B),dim=-1)
print(C)
print(A)


tensor([8, 7, 8, 7, 1])
tensor([4, 7, 6, 9, 4])
tensor([[8, 4],
        [7, 7],
        [8, 6],
        [7, 9],
        [1, 4]])
tensor([8, 7, 8, 7, 1])


In [103]:
import torch
import torch.nn as nn

def precompute_freqs_cis(dim: int, max_seq_len: int, theta: float = 10000.0):
    """
    预计算旋转频率的 cos 和 sin 值
    dim: 注意力头的维度 (head_dim)
    max_seq_len: 最大序列长度
    theta: 基础频率，默认 10000.0 (LLaMA 等模型的默认值)
    """
    # 1. 计算 theta_i: [dim/2]
    # 公式: theta_i = 10000 ^ (-2i/d) => exp(-2i/d * ln(10000))
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))

    # 2. 生成位置索引 m: [max_seq_len]
    t = torch.arange(max_seq_len, dtype=torch.float32)

    # 3. 计算 m * theta_i: 使用外积得到矩阵形状 [max_seq_len, dim/2]
    freqs = torch.outer(t, freqs)

    # 4. 复制两份以匹配原始维度 d: [max_seq_len, dim]
    # 这里对应把前半部分和后半部分结合的写法
    freqs = torch.cat((freqs, freqs), dim=-1)

    # 返回 cos 和 sin (方便后续直接相乘)
    return freqs.cos(), freqs.sin()

def rotate_half(x):
    """
    将向量的后半部分取负，并与前半部分交换位置。
    用于实现公式中的: [-x1, x0] (这里 x 已经被分成了两半)
    """
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, position_ids):
    """
    将 RoPE 应用到 Query 和 Key 上
    q, k 形状: [batch_size, num_heads, seq_len, head_dim]
    """
    # 根据 position_ids 提取对应的 cos 和 sin
    # squeeze(1) 处理是因为 position_ids 通常有 batch 维度，我们通过 unsqueeze 对齐维度
    cos = cos[position_ids].unsqueeze(1)  # [batch_size, 1, seq_len, head_dim]
    sin = sin[position_ids].unsqueeze(1)  # [batch_size, 1, seq_len, head_dim]

    # 核心公式：x * cos(m * theta) + rotate_half(x) * sin(m * theta)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)

    return q_embed, k_embed

# ================= 测试代码 =================
if __name__ == "__main__":
    batch_size = 2
    num_heads = 8
    seq_len = 10
    head_dim = 64
    max_seq_len = 1024

    # 1. 预计算频率 (通常在模型初始化时计算一次即可)
    cos, sin = precompute_freqs_cis(head_dim, max_seq_len)

    # 2. 模拟输入的 Query 和 Key
    q = torch.randn(batch_size, num_heads, seq_len, head_dim)
    k = torch.randn(batch_size, num_heads, seq_len, head_dim)
    print(cos.shape)

    # 3. 模拟位置 ID (通常是 0, 1, 2, ..., seq_len-1)
    position_ids = torch.arange(seq_len).unsqueeze(0).expand(batch_size, -1)

    # 4. 应用 RoPE
    q_out, k_out = apply_rotary_pos_emb(q, k, cos, sin, position_ids)

    print("输入 q shape:", q.shape)
    print("输出 q shape:", q_out.shape)
    print("应用 RoPE 成功！")


torch.Size([1024, 64])
输入 q shape: torch.Size([2, 8, 10, 64])
输出 q shape: torch.Size([2, 8, 10, 64])
应用 RoPE 成功！


In [113]:
torch.tril(torch.ones(3, 3, dtype=torch.bool))

tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])

In [2]:
import torch


def sgd_step(params, grads, lr):
    """
    SGD 单步更新

    Args:
        params: 模型参数列表 (list of nn.Parameter)
        grads:  对应的梯度张量列表
        lr:     学习率
    """
    for p, g in zip(params, grads):
        p.data -= lr * g


# --- 🧪 测试运行 ---
w = torch.tensor([0.0], requires_grad=True)

for step in range(1, 11):
    loss = (w - 3) ** 2
    loss.backward()

    sgd_step([w], [w.grad], lr=0.1)

    w.grad = None   # 清空梯度，避免累积
    if step % 2 == 0:
        print(f"Step {step:2d}: w = {w.item():.4f}, loss = {loss.item():.4f}")

Step  2: w = 1.0800, loss = 5.7600
Step  4: w = 1.7712, loss = 2.3593
Step  6: w = 2.2136, loss = 0.9664
Step  8: w = 2.4967, loss = 0.3958
Step 10: w = 2.6779, loss = 0.1621


In [4]:
import torch


def momentum_step(params, grads, velocities, lr, beta):
    """Momentum 单步更新。velocities 需外部初始化为零"""
    for p, g, v in zip(params, grads, velocities):
        v.data = beta * v.data + (1 - beta) * g
        p.data -= lr * v.data


# --- 测试运行：SGD vs Momentum ---
w_sgd = torch.tensor([0.0], requires_grad=True)
w_mom = torch.tensor([0.0], requires_grad=True)
vel = [torch.zeros_like(w_mom)]

for step in range(1, 31):
    (w_sgd - 3).pow(2).sum().backward()
    (w_mom - 3).pow(2).sum().backward()
    sgd_step([w_sgd], [w_sgd.grad], lr=0.1)
    momentum_step([w_mom], [w_mom.grad], vel, lr=0.1, beta=0.9)
    w_sgd.grad = None
    w_mom.grad = None
    if step % 10 == 0:
        print(f"t={step:<4} SGD: w={w_sgd.item():.4f}  Momentum: w={w_mom.item():.4f}")

print(f"最终 SGD: {w_sgd.item():.4f}, Momentum: {w_mom.item():.4f}")

t=10   SGD: w=2.6779  Momentum: w=2.0786
t=20   SGD: w=2.9654  Momentum: w=3.8120
t=30   SGD: w=2.9963  Momentum: w=3.5275
最终 SGD: 2.9963, Momentum: 3.5275


In [6]:
import torch
import math


def rmsprop_step(params, grads, s_cache, lr, beta2, eps=1e-8):
    """RMSProp 单步更新"""
    for p, g, s in zip(params, grads, s_cache):
        s.data = beta2 * s.data + (1 - beta2) * g.pow(2)
        p.data -= lr * g / (s.data.sqrt() + eps)


# --- 测试运行：不同梯度尺度 ---
w1 = torch.tensor([0.0], requires_grad=True)
w2 = torch.tensor([0.0], requires_grad=True)
s1 = torch.zeros_like(w1)
s2 = torch.zeros_like(w2)

for step in range(1, 6):
    g1 = torch.tensor([50.0])
    g2 = torch.tensor([0.05])
    before1, before2 = w1.item(), w2.item()
    rmsprop_step([w1, w2], [g1, g2], [s1, s2], lr=0.01, beta2=0.999)
    eff1 = 0.01 / (math.sqrt(s1.item()) + 1e-8)
    eff2 = 0.01 / (math.sqrt(s2.item()) + 1e-8)
    print(f"Step {step}: eff_lr₁={eff1:.4f} eff_lr₂={eff2:.2f}  "
          f"Δw₁={w1.item()-before1:+.4f} Δw₂={w2.item()-before2:+.4f}")

Step 1: eff_lr₁=0.0063 eff_lr₂=6.32  Δw₁=-0.3162 Δw₂=-0.3162
Step 2: eff_lr₁=0.0045 eff_lr₂=4.47  Δw₁=-0.2237 Δw₂=-0.2237
Step 3: eff_lr₁=0.0037 eff_lr₂=3.65  Δw₁=-0.1827 Δw₂=-0.1827
Step 4: eff_lr₁=0.0032 eff_lr₂=3.16  Δw₁=-0.1582 Δw₂=-0.1582
Step 5: eff_lr₁=0.0028 eff_lr₂=2.83  Δw₁=-0.1416 Δw₂=-0.1416


In [8]:
import torch
import math


def adam_step(params, grads, m_cache, v_cache, lr, beta1, beta2, eps, t):
    """Adam 单步更新。t 从 1 开始计数"""
    for p, g, m, v in zip(params, grads, m_cache, v_cache):
        m.data = beta1 * m.data + (1 - beta1) * g
        v.data = beta2 * v.data + (1 - beta2) * g.pow(2)
        m_hat = m.data / (1 - beta1 ** t)
        v_hat = v.data / (1 - beta2 ** t)
        p.data -= lr * m_hat / (v_hat.sqrt() + eps)


# --- 测试运行 ---
w = torch.tensor([0.0], requires_grad=True)
m = torch.zeros_like(w)
v = torch.zeros_like(w)

for t in range(1, 11):
    loss = (w - 3) ** 2
    loss.backward()
    adam_step([w], [w.grad], [m], [v], lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, t=t)
    w.grad = None
    m_h = m.item() / (1 - 0.9 ** t)
    v_h = v.item() / (1 - 0.999 ** t)
    step_size = 0.1 * abs(m_h) / (math.sqrt(v_h) + 1e-8)
    if t <= 5:
        print(f"t={t}: w={w.item():.4f}  m̂={m_h:+.4f}  v̂={v_h:.4f}  step={step_size:.4f}")

t=1: w=0.1000  m̂=-6.0000  v̂=36.0000  step=0.1000
t=2: w=0.1999  m̂=-5.8947  v̂=34.8194  step=0.0999
t=3: w=0.2996  m̂=-5.7861  v̂=33.6659  step=0.0997
t=4: w=0.3991  m̂=-5.6740  v̂=32.5398  step=0.0995
t=5: w=0.4982  m̂=-5.5587  v̂=31.4414  step=0.0991


In [10]:
def adamw_step(params, grads, m_cache, v_cache,
               lr, beta1, beta2, eps, t, weight_decay):
    """AdamW 单步更新"""
    for p, g, m, v in zip(params, grads, m_cache, v_cache):
        # 标准 Adam 自适应更新
        m.data = beta1 * m.data + (1 - beta1) * g
        v.data = beta2 * v.data + (1 - beta2) * g.pow(2)
        m_hat = m.data / (1 - beta1 ** t)
        v_hat = v.data / (1 - beta2 ** t)
        p.data -= lr * m_hat / (v_hat.sqrt() + eps)
        # 解耦的权重衰减
        p.data -= lr * weight_decay * p.data


# --- 测试运行：纯衰减对比 ---
import torch

def test(name, use_adamw):
    w = torch.tensor([1.0], requires_grad=True)
    m = torch.zeros_like(w)
    v = torch.zeros_like(w)
    for t in range(1, 51):
        if use_adamw:
            adamw_step([w], [torch.tensor([0.0])], [m], [v],
                       lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8,
                       t=t, weight_decay=0.1)
        else:
            adam_step([w], [torch.tensor([0.1 * w.item()])], [m], [v],
                       lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, t=t)
    print(f"{name:>8}: w 1.00 → {w.item():.4f} (理论 → 0.9512)")

test("Adam-L2", use_adamw=False)
test("AdamW", use_adamw=True)


 Adam-L2: w 1.00 → 0.5368 (理论 → 0.9512)
   AdamW: w 1.00 → 0.9512 (理论 → 0.9512)


In [11]:
def log_decorator(func):            # 1. 接收一个函数
    def wrapper(*args, **kwargs):   # 2. 定义一个包装函数
        print("开始调用", func.__name__)
        result = func(*args, **kwargs) # 3. 调用原函数
        print("结束调用", func.__name__)
        return result
    return wrapper

In [13]:
@log_decorator
def say_hello():
    print("Hello!")
say_hello()

开始调用 say_hello
Hello!
结束调用 say_hello
